# Módulo 1 · Clase 8 — Manejo de archivos

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Bloque | Qué aprenderás |
|---|--------|----------------|
| 1 | Archivos de texto | Leer y escribir ficheros `.txt` con `open()` |
| 2 | Modos de apertura | `r`, `w`, `a`, `x` y sus variantes binarias |
| 3 | `with` | El gestor de contexto que cierra el archivo automáticamente |
| 4 | CSV | Leer y escribir datos tabulares con el módulo `csv` |
| 5 | JSON | Serializar y deserializar datos estructurados con `json` |
| 6 | Rutas y `pathlib` | Trabajar con rutas de forma robusta y multiplataforma |
| 7 | Práctica | Analizador CSV |

> 💡 **¿Por qué es importante?** Casi toda la IA y el análisis de datos empieza leyendo un archivo. Los datasets son CSV o JSON. Los modelos se guardan en disco. Los logs son ficheros de texto. Saber leer y escribir archivos es una habilidad fundamental.

---
## Preparación — Creamos los archivos de ejemplo

Ejecuta esta celda primero para generar los ficheros que usaremos durante la clase.

In [ ]:
import os

# Directorio de trabajo para esta clase
os.makedirs("archivos_clase8", exist_ok=True)

# Archivo de texto de ejemplo
with open("archivos_clase8/poema.txt", "w", encoding="utf-8") as f:
    f.write("""Caminante, son tus huellas
el camino y nada más;
caminante, no hay camino,
se hace camino al andar.
Al andar se hace camino
y al volver la vista atrás
se ve la senda que nunca
se ha de volver a pisar.
Caminante, no hay camino,
sino estelas en la mar.
""")

# CSV de ejemplo: ventas
with open("archivos_clase8/ventas.csv", "w", encoding="utf-8") as f:
    f.write("""fecha,producto,categoria,cantidad,precio_unitario,vendedor
2024-01-03,Laptop,Electrónica,2,899.99,Ana
2024-01-05,Ratón,Electrónica,10,25.50,Luis
2024-01-08,Silla,Mobiliario,3,199.00,Ana
2024-01-12,Monitor,Electrónica,1,349.00,María
2024-01-15,Teclado,Electrónica,5,79.99,Luis
2024-01-18,Mesa,Mobiliario,2,299.00,María
2024-01-20,Laptop,Electrónica,1,899.99,Ana
2024-01-22,Auriculares,Electrónica,8,59.99,Luis
2024-01-25,Estantería,Mobiliario,4,149.00,Ana
2024-02-01,Monitor,Electrónica,2,349.00,María
2024-02-05,Ratón,Electrónica,15,25.50,Luis
2024-02-10,Silla,Mobiliario,6,199.00,Ana
2024-02-14,Laptop,Electrónica,3,899.99,María
2024-02-18,Teclado,Electrónica,7,79.99,Luis
2024-02-22,Mesa,Mobiliario,1,299.00,Ana
""")

# JSON de ejemplo: configuración
import json
config = {
    "app": "AnalizadorCSV",
    "version": "1.0",
    "configuracion": {
        "separador": ",",
        "encoding": "utf-8",
        "decimales": 2
    },
    "columnas_numericas": ["cantidad", "precio_unitario"],
    "activo": True
}
with open("archivos_clase8/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4, ensure_ascii=False)

print("✅ Archivos creados en archivos_clase8/")
print(os.listdir("archivos_clase8"))

---
## 1. Archivos de texto — `open()`

La función `open()` abre un archivo y devuelve un **objeto de archivo** con el que podemos leer o escribir.

```python
archivo = open("ruta/al/archivo.txt", modo, encoding="utf-8")
# ... operaciones ...
archivo.close()   # SIEMPRE hay que cerrar el archivo
```

### Modos de apertura

| Modo | Nombre | Qué hace | Si el archivo no existe |
|------|--------|----------|-------------------------|
| `"r"` | Read | Solo lectura | Error (`FileNotFoundError`) |
| `"w"` | Write | Escritura — **borra** el contenido existente | Lo crea |
| `"a"` | Append | Añade al final sin borrar | Lo crea |
| `"x"` | Exclusive | Crea el archivo | Error si ya existe |
| `"r+"` | Read+Write | Lectura y escritura | Error si no existe |

> ⚠️ Añade `"b"` para modo binario: `"rb"`, `"wb"` — para imágenes, PDFs, etc.

> 🌍 Usa siempre `encoding="utf-8"` para evitar problemas con caracteres especiales (tildes, ñ…).

---
## 2. El gestor de contexto `with`

La forma correcta de abrir archivos en Python es con `with`. Garantiza que el archivo se cierre automáticamente aunque ocurra un error.

```python
# Sin with — peligroso: si falla, el archivo queda abierto
archivo = open("datos.txt", "r")
contenido = archivo.read()
archivo.close()   # puede que nunca llegue aquí

# Con with — el archivo siempre se cierra al salir del bloque
with open("datos.txt", "r", encoding="utf-8") as archivo:
    contenido = archivo.read()
# aquí el archivo ya está cerrado
```

---
## 3. Lectura de archivos de texto

Hay tres métodos de lectura:

| Método | Devuelve | Cuándo usarlo |
|--------|----------|---------------|
| `.read()` | `str` con todo el contenido | Archivos pequeños |
| `.readline()` | `str` con una sola línea | Línea a línea |
| `.readlines()` | `list` de strings (una por línea) | Cuando necesitas la lista completa |
| Iterar el archivo | Una línea por iteración | Archivos grandes (no carga todo en memoria) |

In [ ]:
# .read() — todo el contenido como un string
with open("archivos_clase8/poema.txt", "r", encoding="utf-8") as f:
    contenido = f.read()

print(type(contenido))
print(f"Caracteres: {len(contenido)}")
print("---")
print(contenido)

In [ ]:
# .readlines() — lista de líneas (incluye el \n al final de cada una)
with open("archivos_clase8/poema.txt", "r", encoding="utf-8") as f:
    lineas = f.readlines()

print(f"Número de líneas: {len(lineas)}")
print(f"Primera línea: {repr(lineas[0])}")   # repr() muestra el \n

# Limpiar el \n con strip()
lineas_limpias = [l.strip() for l in lineas]
print(f"Primera limpia: {repr(lineas_limpias[0])}")

In [ ]:
# Iterar el archivo — más eficiente para archivos grandes
# (no carga todo en memoria a la vez)
print("Líneas que contienen 'camino':")
with open("archivos_clase8/poema.txt", "r", encoding="utf-8") as f:
    for numero, linea in enumerate(f, start=1):
        if "camino" in linea.lower():
            print(f"  Línea {numero}: {linea.strip()}")

---
## 4. Escritura de archivos de texto

In [ ]:
# Modo "w" — crea o sobreescribe
with open("archivos_clase8/notas.txt", "w", encoding="utf-8") as f:
    f.write("Primera línea\n")
    f.write("Segunda línea\n")
    f.writelines(["Tercera\n", "Cuarta\n", "Quinta\n"])  # escribe una lista

# Verificar
with open("archivos_clase8/notas.txt", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Modo "a" — añade al final sin borrar el contenido
with open("archivos_clase8/notas.txt", "a", encoding="utf-8") as f:
    f.write("Sexta línea (añadida)\n")
    f.write("Séptima línea (añadida)\n")

with open("archivos_clase8/notas.txt", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Caso real: escribir un log de eventos
from datetime import datetime

def escribir_log(ruta, nivel, mensaje):
    """Añade una entrada al archivo de log."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entrada   = f"[{timestamp}] [{nivel.upper()}] {mensaje}\n"
    with open(ruta, "a", encoding="utf-8") as f:
        f.write(entrada)

log_path = "archivos_clase8/app.log"
escribir_log(log_path, "info",  "Aplicación iniciada")
escribir_log(log_path, "info",  "Cargando datos...")
escribir_log(log_path, "error", "No se encontró el archivo ventas_feb.csv")
escribir_log(log_path, "info",  "Usando datos de caché")

with open(log_path, "r", encoding="utf-8") as f:
    print(f.read())

---
## 5. CSV — Comma-Separated Values

Un CSV es un archivo de texto donde cada línea es una fila y los valores van separados por comas (u otro delimitador). Es el formato más usado para intercambiar datos tabulares.

```
fecha,producto,cantidad,precio
2024-01-03,Laptop,2,899.99
2024-01-05,Ratón,10,25.50
```

Python tiene el módulo `csv` en la librería estándar — no necesitas instalar nada.

### ¿Por qué no simplemente `.split(",")`?

Un CSV puede contener comas dentro de valores entre comillas:
```
"García, Ana",28,"Madrid, España"
```
El módulo `csv` maneja estos casos correctamente; `.split(",")` no.

In [ ]:
import csv

# csv.reader — devuelve cada fila como una lista de strings
print("=== csv.reader ===")
with open("archivos_clase8/ventas.csv", "r", encoding="utf-8") as f:
    lector = csv.reader(f)
    cabecera = next(lector)   # primera fila = nombres de columnas
    print(f"Cabecera: {cabecera}")
    print()
    for i, fila in enumerate(lector):
        print(fila)
        if i == 2: break   # mostramos solo las primeras 3 filas

In [ ]:
# csv.DictReader — devuelve cada fila como un diccionario
# La clave es el nombre de la columna — mucho más cómodo
print("=== csv.DictReader ===")
with open("archivos_clase8/ventas.csv", "r", encoding="utf-8") as f:
    lector = csv.DictReader(f)
    filas  = list(lector)   # cargamos todas las filas

print(f"Total de filas: {len(filas)}")
print(f"Columnas:       {list(filas[0].keys())}")
print()

# Acceso por nombre de columna
for fila in filas[:3]:
    print(f"{fila['fecha']}  {fila['producto']:15} "
          f"x{fila['cantidad']}  {float(fila['precio_unitario']):.2f}€")

In [ ]:
# csv.writer — escribir un CSV
alumnos = [
    {"nombre": "Ana García",  "nota": 8.5, "aprobado": True},
    {"nombre": "Luis Pérez",  "nota": 4.0, "aprobado": False},
    {"nombre": "María López", "nota": 7.0, "aprobado": True},
]

# csv.writer
with open("archivos_clase8/alumnos.csv", "w", encoding="utf-8", newline="") as f:
    escritor = csv.writer(f)
    escritor.writerow(["nombre", "nota", "aprobado"])   # cabecera
    for a in alumnos:
        escritor.writerow([a["nombre"], a["nota"], a["aprobado"]])

# csv.DictWriter — más elegante cuando tienes dicts
campos = ["nombre", "nota", "aprobado"]
with open("archivos_clase8/alumnos_dict.csv", "w",
          encoding="utf-8", newline="") as f:
    escritor = csv.DictWriter(f, fieldnames=campos)
    escritor.writeheader()    # escribe la cabecera automáticamente
    escritor.writerows(alumnos)

# Verificar
with open("archivos_clase8/alumnos_dict.csv", "r", encoding="utf-8") as f:
    print(f.read())

> ⚠️ **`newline=""`** al escribir CSVs en Windows evita que se inserten líneas en blanco extra entre filas. Es una peculiaridad del módulo `csv` — ponlo siempre al escribir.

---
## 6. JSON — JavaScript Object Notation

JSON es el formato estándar para intercambiar datos estructurados entre sistemas (APIs web, configuraciones, bases de datos NoSQL…). Su sintaxis es muy similar a los diccionarios de Python.

```json
{
    "nombre": "Ana",
    "edad": 28,
    "activo": true,
    "etiquetas": ["python", "data"],
    "direccion": null
}
```

### Equivalencias Python ↔ JSON

| Python | JSON |
|--------|------|
| `dict` | object `{}` |
| `list`, `tuple` | array `[]` |
| `str` | string `""` |
| `int`, `float` | number |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

### Las cuatro funciones del módulo `json`

| Función | Qué hace | Dirección |
|---------|----------|----------|
| `json.dump(obj, f)` | Python → JSON en archivo | Python → Disco |
| `json.load(f)` | JSON en archivo → Python | Disco → Python |
| `json.dumps(obj)` | Python → string JSON | Python → String |
| `json.loads(texto)` | String JSON → Python | String → Python |

In [ ]:
import json

# json.load — leer un JSON desde archivo
with open("archivos_clase8/config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

print(type(config))           # dict
print(config["app"])          # AnalizadorCSV
print(config["version"])      # 1.0
print(config["configuracion"]["decimales"])   # 2
print(config["columnas_numericas"])           # ['cantidad', 'precio_unitario']
print()

# El objeto Python ya está listo para usar
for clave, valor in config["configuracion"].items():
    print(f"  {clave}: {valor}")

In [ ]:
# json.dump — escribir un JSON en archivo
resultado_analisis = {
    "archivo": "ventas.csv",
    "total_filas": 15,
    "total_ventas": 12345.67,
    "productos": ["Laptop", "Ratón", "Silla"],
    "completo": True,
    "errores": None
}

with open("archivos_clase8/resultado.json", "w", encoding="utf-8") as f:
    json.dump(
        resultado_analisis,
        f,
        indent=4,            # indentación legible
        ensure_ascii=False   # permite tildes y caracteres especiales
    )

# Verificar
with open("archivos_clase8/resultado.json", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# json.dumps y json.loads — trabajar con strings (sin archivos)
# Muy útil cuando recibes datos de una API o los envías

datos = {"nombre": "Ana", "puntuacion": 9.5, "aprobado": True}

# Python → string JSON
texto_json = json.dumps(datos, ensure_ascii=False)
print(f"String JSON: {texto_json}")
print(f"Tipo:        {type(texto_json)}")

# String JSON → Python
recuperado = json.loads(texto_json)
print(f"\nRecuperado:  {recuperado}")
print(f"Tipo:        {type(recuperado)}")
print(f"Acceso:      {recuperado['nombre']}")

---
## 7. `pathlib` — Rutas de forma robusta

`pathlib` es la forma moderna de trabajar con rutas de archivos en Python. Resuelve el problema de las rutas que funcionan en Mac/Linux (`/`) pero no en Windows (`\`).

```python
from pathlib import Path

ruta = Path("archivos_clase8") / "ventas.csv"   # / construye rutas
```

In [ ]:
from pathlib import Path

# Construir rutas de forma multiplataforma
carpeta = Path("archivos_clase8")
archivo = carpeta / "ventas.csv"

print(f"Ruta:       {archivo}")
print(f"Nombre:     {archivo.name}")
print(f"Extensión:  {archivo.suffix}")
print(f"Sin ext:    {archivo.stem}")
print(f"Carpeta:    {archivo.parent}")
print(f"¿Existe?    {archivo.exists()}")
print(f"¿Es file?   {archivo.is_file()}")
print(f"Tamaño:     {archivo.stat().st_size} bytes")

# Listar archivos de un directorio
print("\nArchivos en archivos_clase8/:")
for f in sorted(carpeta.iterdir()):
    print(f"  {f.name:30} {f.stat().st_size:6} bytes")

In [ ]:
# pathlib integra lectura y escritura directamente
ruta = Path("archivos_clase8") / "nota_rapida.txt"

# Escribir directamente con Path
ruta.write_text("Esto es una nota rápida.\n", encoding="utf-8")

# Leer directamente con Path
contenido = ruta.read_text(encoding="utf-8")
print(contenido)

# Buscar archivos por patrón
print("Archivos CSV:")
for csv_file in Path("archivos_clase8").glob("*.csv"):
    print(f"  {csv_file.name}")

---
## 8. Manejo de errores con archivos

Los archivos son una fuente habitual de errores: el archivo no existe, no tenemos permisos, el disco está lleno… Siempre debemos anticiparlos.

In [ ]:
# Los errores más comunes con archivos
from pathlib import Path

def leer_archivo_seguro(ruta):
    """Lee un archivo y maneja los errores más comunes."""
    try:
        with open(ruta, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"Error: el archivo '{ruta}' no existe.")
        return None
    except PermissionError:
        print(f"Error: no tienes permiso para leer '{ruta}'.")
        return None
    except UnicodeDecodeError:
        print(f"Error: el archivo '{ruta}' no está en UTF-8. Prueba con encoding='latin-1'.")
        return None

# Prueba con archivo existente
contenido = leer_archivo_seguro("archivos_clase8/notas.txt")
if contenido:
    print(f"Leído: {len(contenido)} caracteres")

# Prueba con archivo inexistente
leer_archivo_seguro("no_existe.txt")

---
## 9. Práctica — Analizador CSV

Vamos a construir un analizador del fichero `ventas.csv` que generamos al principio. Aplicamos todo lo aprendido: leer CSV, procesar datos, guardar resultados en JSON y generar un informe en texto.

---

### Ejercicio 1 — Cargar y limpiar el CSV

**Enunciado:** Escribe una función `cargar_ventas(ruta)` que:
1. Lea el CSV con `DictReader`.
2. Convierta `cantidad` a `int` y `precio_unitario` a `float`.
3. Añada una columna calculada `total` = `cantidad × precio_unitario`.
4. Devuelva la lista de diccionarios limpia.

**Salida esperada (primeras 3 filas):**
```
{'fecha': '2024-01-03', 'producto': 'Laptop', 'categoria': 'Electrónica',
 'cantidad': 2, 'precio_unitario': 899.99, 'vendedor': 'Ana', 'total': 1799.98}
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import csv
from pathlib import Path

RUTA_CSV = Path("archivos_clase8") / "ventas.csv"


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import csv
from pathlib import Path

RUTA_CSV = Path("archivos_clase8") / "ventas.csv"

def cargar_ventas(ruta):
    """Carga el CSV de ventas, convierte tipos y añade columna 'total'."""
    ventas = []
    with open(ruta, "r", encoding="utf-8") as f:
        for fila in csv.DictReader(f):
            fila["cantidad"]        = int(fila["cantidad"])
            fila["precio_unitario"] = float(fila["precio_unitario"])
            fila["total"]           = round(fila["cantidad"] * fila["precio_unitario"], 2)
            ventas.append(fila)
    return ventas

ventas = cargar_ventas(RUTA_CSV)
print(f"Filas cargadas: {len(ventas)}")
for v in ventas[:3]:
    print(v)
```

**Puntos clave:**
- `DictReader` devuelve todos los valores como strings — la conversión de tipos es el primer paso obligatorio.
- Modificamos el dict directamente en el bucle: `fila["cantidad"] = int(...)` — no necesitamos crear un dict nuevo.
- `round(..., 2)` evita errores de punto flotante en la columna `total`.
</details>

---
### Ejercicio 2 — Estadísticas globales

**Enunciado:** Escribe una función `estadisticas_globales(ventas)` que devuelva un diccionario con:
- `total_filas`: número de registros
- `total_facturado`: suma de todos los `total`
- `ticket_medio`: media del campo `total`
- `venta_maxima`: la fila con mayor `total` (dict completo)
- `venta_minima`: la fila con menor `total` (dict completo)
- `productos_unicos`: set de productos distintos
- `vendedores`: set de vendedores

**Salida esperada:**
```
Total filas:       15
Total facturado:   10.247,82 €
Ticket medio:      683,19 €
Venta máxima:      Laptop × 3  →  2.699,97 € (María)
Productos únicos:  6
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Usa la lista 'ventas' del ejercicio anterior


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
def estadisticas_globales(ventas):
    """Calcula estadísticas generales del dataset de ventas."""
    totales = [v["total"] for v in ventas]

    return {
        "total_filas":       len(ventas),
        "total_facturado":   round(sum(totales), 2),
        "ticket_medio":      round(sum(totales) / len(totales), 2),
        "venta_maxima":      max(ventas, key=lambda v: v["total"]),
        "venta_minima":      min(ventas, key=lambda v: v["total"]),
        "productos_unicos":  {v["producto"] for v in ventas},
        "vendedores":        {v["vendedor"] for v in ventas}
    }

stats = estadisticas_globales(ventas)

max_v = stats["venta_maxima"]
min_v = stats["venta_minima"]

print(f"Total filas:       {stats['total_filas']}")
print(f"Total facturado:   {stats['total_facturado']:,.2f} €")
print(f"Ticket medio:      {stats['ticket_medio']:,.2f} €")
print(f"Venta máxima:      {max_v['producto']} × {max_v['cantidad']} "
      f" →  {max_v['total']:,.2f} € ({max_v['vendedor']})")
print(f"Venta mínima:      {min_v['producto']} × {min_v['cantidad']} "
      f" →  {min_v['total']:,.2f} € ({min_v['vendedor']})")
print(f"Productos únicos:  {len(stats['productos_unicos'])} → {sorted(stats['productos_unicos'])}")
print(f"Vendedores:        {sorted(stats['vendedores'])}")
```

**Puntos clave:**
- `{v["producto"] for v in ventas}` es una **set comprehension** — construye el set de productos únicos en una línea.
- `max(ventas, key=lambda v: v["total"])` devuelve el dict completo de la venta más alta — `key` dice por qué criterio comparar.
- `{stats['total_facturado']:,.2f}` usa el formato `,` para separador de miles y `.2f` para dos decimales.
</details>

---
### Ejercicio 3 — Análisis por agrupación

**Enunciado:** Escribe funciones para agrupar y resumir las ventas:

1. `ventas_por_vendedor(ventas)` → dict `{vendedor: total_facturado}`
2. `ventas_por_categoria(ventas)` → dict `{categoria: total_facturado}`
3. `ventas_por_producto(ventas)` → dict `{producto: {"unidades": int, "facturado": float}}`

Muestra los resultados ordenados de mayor a menor facturación.

**Salida esperada:**
```
=== Por vendedor ===
María   3.947,96 €
Ana     3.894,94 €
Luis    2.404,92 €

=== Por categoría ===
Electrónica   8.533,86 €
Mobiliario    1.713,96 €
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
def ventas_por_vendedor(ventas):
    resultado = {}
    for v in ventas:
        clave = v["vendedor"]
        resultado[clave] = round(resultado.get(clave, 0) + v["total"], 2)
    return dict(sorted(resultado.items(), key=lambda x: x[1], reverse=True))

def ventas_por_categoria(ventas):
    resultado = {}
    for v in ventas:
        clave = v["categoria"]
        resultado[clave] = round(resultado.get(clave, 0) + v["total"], 2)
    return dict(sorted(resultado.items(), key=lambda x: x[1], reverse=True))

def ventas_por_producto(ventas):
    resultado = {}
    for v in ventas:
        clave = v["producto"]
        if clave not in resultado:
            resultado[clave] = {"unidades": 0, "facturado": 0.0}
        resultado[clave]["unidades"]  += v["cantidad"]
        resultado[clave]["facturado"] = round(
            resultado[clave]["facturado"] + v["total"], 2)
    return dict(sorted(resultado.items(),
                       key=lambda x: x[1]["facturado"], reverse=True))

# Mostrar vendedores
print("=== Por vendedor ===")
for vendedor, total in ventas_por_vendedor(ventas).items():
    print(f"  {vendedor:<10} {total:>10,.2f} €")

# Mostrar categorías
print("\n=== Por categoría ===")
for cat, total in ventas_por_categoria(ventas).items():
    print(f"  {cat:<15} {total:>10,.2f} €")

# Mostrar productos
print("\n=== Por producto ===")
for prod, datos in ventas_por_producto(ventas).items():
    print(f"  {prod:<15} {datos['unidades']:>4} uds  {datos['facturado']:>10,.2f} €")
```

**Puntos clave:**
- `resultado.get(clave, 0)` devuelve 0 si la clave no existe todavía — evita `KeyError` y elimina la necesidad de inicializar cada entrada.
- `sorted(resultado.items(), key=lambda x: x[1], reverse=True)` ordena el dict por valor de mayor a menor.
- `{vendedor:<10}` alinea a la izquierda en 10 caracteres; `{total:>10,.2f}` alinea a la derecha con separador de miles.
</details>

---
### Ejercicio 4 — Guardar resultados en JSON y CSV

**Enunciado:** Escribe dos funciones:

1. `guardar_resumen_json(stats, ruta)` — guarda el diccionario de estadísticas en un archivo JSON. Convierte los `set` a listas ordenadas para que sea serializable.
2. `guardar_resumen_csv(agrupacion, ruta, columnas)` — guarda cualquiera de los diccionarios de agrupación como CSV.

**Archivos esperados:**
- `archivos_clase8/resumen.json`
- `archivos_clase8/ventas_por_vendedor.csv`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import json
import csv
from pathlib import Path

def guardar_resumen_json(stats, ruta):
    """Guarda el diccionario de estadísticas en JSON.
    Convierte sets y dicts de ventas a tipos serializables.
    """
    def serializable(obj):
        """Convierte tipos no serializables."""
        if isinstance(obj, set):
            return sorted(obj)
        if isinstance(obj, dict):
            return {k: serializable(v) for k, v in obj.items()}
        return obj

    resumen = {
        "total_filas":      stats["total_filas"],
        "total_facturado":  stats["total_facturado"],
        "ticket_medio":     stats["ticket_medio"],
        "productos_unicos": sorted(stats["productos_unicos"]),
        "vendedores":       sorted(stats["vendedores"]),
        "venta_maxima":     dict(stats["venta_maxima"]),
        "venta_minima":     dict(stats["venta_minima"])
    }
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(resumen, f, indent=4, ensure_ascii=False)
    print(f"✅ Resumen guardado en {ruta}")

def guardar_resumen_csv(agrupacion, ruta, col_clave, col_valor):
    """Guarda un dict {clave: valor} como CSV con dos columnas."""
    with open(ruta, "w", encoding="utf-8", newline="") as f:
        escritor = csv.writer(f)
        escritor.writerow([col_clave, col_valor])
        for clave, valor in agrupacion.items():
            escritor.writerow([clave, valor])
    print(f"✅ CSV guardado en {ruta}")

# Uso
stats = estadisticas_globales(ventas)
guardar_resumen_json(stats, Path("archivos_clase8") / "resumen.json")

por_vendedor = ventas_por_vendedor(ventas)
guardar_resumen_csv(
    por_vendedor,
    Path("archivos_clase8") / "ventas_por_vendedor.csv",
    "vendedor", "total_facturado"
)

# Verificar JSON
with open(Path("archivos_clase8") / "resumen.json", "r", encoding="utf-8") as f:
    print(f.read())
```

**Puntos clave:**
- `json.dump` falla si hay tipos no serializables — los `set` de Python no existen en JSON. Hay que convertirlos a listas con `sorted()`.
- La función `serializable()` anidada es un helper local: solo la necesita `guardar_resumen_json`, así que vive dentro de ella.
- `newline=""` al abrir el CSV para escritura es obligatorio en Windows para evitar líneas en blanco.
</details>

---
### Ejercicio 5 (reto) — Informe completo en texto

**Enunciado:** Escribe una función `generar_informe(ventas, ruta_salida)` que genere un informe completo en texto plano y lo guarde en `archivos_clase8/informe.txt`. El informe debe incluir:

1. Encabezado con fecha y nombre del archivo analizado.
2. Estadísticas globales.
3. Ranking de vendedores (de mayor a menor facturación).
4. Detalle por producto (unidades vendidas y facturado).
5. Pie con total de secciones generadas.

**Vista previa del informe:**
```
================================================
   INFORME DE VENTAS — 2024-03-01 10:30:15
================================================
Archivo analizado: ventas.csv
Registros totales: 15

--- RESUMEN GLOBAL ---
Total facturado:  10.247,82 €
Ticket medio:        683,19 €
...
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from datetime import datetime


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
from datetime import datetime
from pathlib import Path

def generar_informe(ventas, ruta_salida):
    """Genera un informe de ventas completo en texto plano."""
    stats        = estadisticas_globales(ventas)
    por_vendedor = ventas_por_vendedor(ventas)
    por_producto = ventas_por_producto(ventas)
    por_categoria = ventas_por_categoria(ventas)

    ahora  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    sep    = "=" * 50
    subsep = "-" * 50
    lineas = []

    # Encabezado
    lineas += [
        sep,
        f"   INFORME DE VENTAS — {ahora}",
        sep,
        f"Archivo analizado: {RUTA_CSV.name}",
        f"Registros totales: {stats['total_filas']}",
        f"Productos únicos:  {len(stats['productos_unicos'])}",
        f"Vendedores:        {', '.join(sorted(stats['vendedores']))}",
        ""
    ]

    # Resumen global
    max_v = stats["venta_maxima"]
    min_v = stats["venta_minima"]
    lineas += [
        f"{'--- RESUMEN GLOBAL ':—<50}",
        f"Total facturado:  {stats['total_facturado']:>10,.2f} €",
        f"Ticket medio:     {stats['ticket_medio']:>10,.2f} €",
        f"Venta máxima:     {max_v['producto']} ×{max_v['cantidad']}"
        f"  =  {max_v['total']:,.2f} € ({max_v['vendedor']})",
        f"Venta mínima:     {min_v['producto']} ×{min_v['cantidad']}"
        f"  =  {min_v['total']:,.2f} € ({min_v['vendedor']})",
        ""
    ]

    # Ranking vendedores
    lineas.append(f"{'--- RANKING VENDEDORES ':—<50}")
    for rango, (vendedor, total) in enumerate(por_vendedor.items(), start=1):
        lineas.append(f"  {rango}. {vendedor:<12} {total:>10,.2f} €")
    lineas.append("")

    # Detalle por categoría
    lineas.append(f"{'--- POR CATEGORÍA ':—<50}")
    for cat, total in por_categoria.items():
        lineas.append(f"  {cat:<20} {total:>10,.2f} €")
    lineas.append("")

    # Detalle por producto
    lineas.append(f"{'--- DETALLE POR PRODUCTO ':—<50}")
    lineas.append(f"  {'Producto':<15} {'Unidades':>9} {'Facturado':>12}")
    lineas.append(f"  {'-'*15} {'-'*9} {'-'*12}")
    for prod, datos in por_producto.items():
        lineas.append(
            f"  {prod:<15} {datos['unidades']:>9} {datos['facturado']:>11,.2f} €"
        )
    lineas.append("")

    # Pie
    lineas += [
        sep,
        f"Informe generado el {ahora}",
        f"Secciones: Resumen global | Vendedores | Categorías | Productos",
        sep
    ]

    # Escribir el archivo
    contenido = "\n".join(lineas)
    with open(ruta_salida, "w", encoding="utf-8") as f:
        f.write(contenido)

    print(f"✅ Informe guardado en {ruta_salida}")
    print(contenido)

generar_informe(ventas, Path("archivos_clase8") / "informe.txt")
```

**Puntos clave:**
- Acumular las líneas en una lista y unirlas al final con `"\n".join(lineas)` es más eficiente que concatenar strings en un bucle.
- `f"{'--- RESUMEN GLOBAL ':—<50}"` usa relleno de formato: el texto se alinea a la izquierda y el resto de los 50 caracteres se rellena con `—`.
- Separar el cálculo (funciones de estadísticas) de la presentación (informe) sigue el principio de responsabilidad única — si cambia el formato, no tocamos la lógica.
- El informe se puede leer directamente en un editor o enviarse por email sin necesidad de software especial.
</details>

---
## Resumen de la clase

### Archivos de texto

```python
# Leer
with open("ruta.txt", "r", encoding="utf-8") as f:
    todo    = f.read()          # string completo
    lineas  = f.readlines()     # lista de strings
    for linea in f: ...         # iteración eficiente

# Escribir
with open("ruta.txt", "w", encoding="utf-8") as f:   # sobreescribe
    f.write("texto\n")

with open("ruta.txt", "a", encoding="utf-8") as f:   # añade
    f.write("más texto\n")
```

### CSV

```python
import csv

# Leer
with open("datos.csv", "r", encoding="utf-8") as f:
    for fila in csv.DictReader(f):     # fila es un dict
        print(fila["columna"])

# Escribir
with open("salida.csv", "w", encoding="utf-8", newline="") as f:
    escritor = csv.DictWriter(f, fieldnames=["a", "b"])
    escritor.writeheader()
    escritor.writerows(lista_de_dicts)
```

### JSON

```python
import json

# Leer desde archivo
with open("datos.json", "r", encoding="utf-8") as f:
    datos = json.load(f)

# Escribir en archivo
with open("salida.json", "w", encoding="utf-8") as f:
    json.dump(datos, f, indent=4, ensure_ascii=False)

# En memoria
texto  = json.dumps(datos)     # Python → string
datos  = json.loads(texto)     # string → Python
```

### pathlib

```python
from pathlib import Path

ruta = Path("carpeta") / "archivo.csv"   # construir ruta
ruta.exists()                             # ¿existe?
ruta.stat().st_size                       # tamaño en bytes
ruta.read_text(encoding="utf-8")          # leer directamente
ruta.write_text("texto", encoding="utf-8") # escribir directamente
Path("carpeta").glob("*.csv")             # buscar por patrón
```

---

## ¿Qué viene en la próxima clase?

En la **Clase 9** aprenderemos a manejar **errores y excepciones** de forma profesional:
- `try`, `except`, `else`, `finally`
- Tipos de excepciones en Python
- Crear excepciones propias con `raise`

Con esto podremos hacer que el analizador CSV maneje archivos corruptos, columnas que faltan o datos malformados sin que el programa se rompa.